# 09 — Final Strategy vs BIST100

Bu notebook final Robot stratejisini aynı tarih aralıklarında BIST100
(`XU100.IS`) ile karşılaştırır.

Üç seri kullanılır:

- **Final Strategy:** Komisyon ve slippage dahil Robot portföyü
- **BIST100 Gross:** Endeks kapanışının 500.000 TL'ye normalize edilmiş hali
- **BIST100 Net:** İlk açılışta alım ve son kapanışta satış; aynı komisyon ve
  slippage varsayımları uygulanır

Karşılaştırma dönemleri:

- Development: 2018–2022
- Validation: 2023–2024
- Holdout: 2025–son veri
- Full: 2018–son veri


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.features import add_indicators
from src.signals import build_market_regime
from src.experiments import evaluate_strategy
from src.metrics import portfolio_metrics
from src.presets import (
    FINAL_STRATEGY_CONFIG,
    FINAL_PORTFOLIO_CONFIG,
)
from src.benchmark import (
    build_benchmark_equity,
    align_equity_curves,
    active_performance_metrics,
    drawdown_series,
    calendar_return_table,
)


## 1. Verileri yükle ve final stratejiyi hazırla


In [ ]:
stock_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bist100_robot_clean.parquet"
)

market_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xu100_robot_clean.parquet"
)

stock_features = add_indicators(stock_prices)
market_features = add_indicators(market_prices)
market_regime = build_market_regime(market_features)

LAST_DATE = min(
    stock_features["Date"].max(),
    market_prices["Date"].max(),
).strftime("%Y-%m-%d")

PERIODS = {
    "Development": ("2018-01-01", "2022-12-31"),
    "Validation": ("2023-01-01", "2024-12-31"),
    "Holdout": ("2025-01-01", LAST_DATE),
    "Full": ("2018-01-01", LAST_DATE),
}

print("Son ortak veri tarihi:", LAST_DATE)
print("Final strateji:", FINAL_STRATEGY_CONFIG)
print("Final portföy:", FINAL_PORTFOLIO_CONFIG)


## 2. Strateji ve BIST100 performanslarını dönem bazında hesapla

BIST100 Gross standart endeks karşılaştırmasıdır. BIST100 Net ise tek seferlik
alım-satım maliyetini de içerir.


In [ ]:
period_records = []
active_records = []
stored_outputs = {}

empty_trades = pd.DataFrame(columns=["Return"])

for period_name, (start, end) in PERIODS.items():
    (
        strategy_raw_metrics,
        strategy_equity,
        strategy_trades,
    ) = evaluate_strategy(
        stock_features=stock_features,
        market_regime=market_regime,
        strategy_config=FINAL_STRATEGY_CONFIG,
        portfolio_config=FINAL_PORTFOLIO_CONFIG,
        start=start,
        end=end,
    )

    benchmark_gross = build_benchmark_equity(
        market_prices=market_prices,
        comparison_dates=strategy_equity["Date"],
        initial_capital=(
            FINAL_PORTFOLIO_CONFIG.initial_capital
        ),
        include_costs=False,
        benchmark_name="BIST100",
    )

    benchmark_net = build_benchmark_equity(
        market_prices=market_prices,
        comparison_dates=strategy_equity["Date"],
        initial_capital=(
            FINAL_PORTFOLIO_CONFIG.initial_capital
        ),
        commission_rate=(
            FINAL_PORTFOLIO_CONFIG.commission_rate
        ),
        slippage_rate=(
            FINAL_PORTFOLIO_CONFIG.slippage_rate
        ),
        include_costs=True,
        benchmark_name="BIST100",
    )

    common_dates = set(benchmark_gross["Date"])

    strategy_aligned = (
        strategy_equity.loc[
            strategy_equity["Date"].isin(common_dates)
        ]
        .sort_values("Date")
        .reset_index(drop=True)
    )

    strategy_metrics = portfolio_metrics(
        strategy_aligned,
        strategy_trades,
    )
    gross_metrics = portfolio_metrics(
        benchmark_gross,
        empty_trades,
    )
    net_metrics = portfolio_metrics(
        benchmark_net,
        empty_trades,
    )

    metric_map = {
        "Final Strategy": strategy_metrics,
        "BIST100 Gross": gross_metrics,
        "BIST100 Net": net_metrics,
    }

    for portfolio_name, metrics in metric_map.items():
        period_records.append(
            {
                "Period": period_name,
                "Portfolio": portfolio_name,
                "Start_Date": (
                    strategy_aligned["Date"].min()
                ),
                "End_Date": (
                    strategy_aligned["Date"].max()
                ),
                "Start_Value_TL": metrics["Start_Value"],
                "End_Value_TL": metrics["End_Value"],
                "Total_Return_%": metrics[
                    "Total_Return_%"
                ],
                "CAGR_%": metrics["CAGR_%"],
                "Max_Drawdown_%": metrics[
                    "Max_Drawdown_%"
                ],
                "Sharpe": metrics["Sharpe"],
                "Sortino": metrics["Sortino"],
                "Calmar": metrics["Calmar"],
                "Profit_Factor": (
                    metrics["Profit_Factor"]
                    if portfolio_name == "Final Strategy"
                    else np.nan
                ),
                "Win_Rate_%": (
                    metrics["Win_Rate_%"]
                    if portfolio_name == "Final Strategy"
                    else np.nan
                ),
                "Trade_Count": (
                    metrics["Trade_Count"]
                    if portfolio_name == "Final Strategy"
                    else np.nan
                ),
                "Exposure_%": metrics["Exposure_%"],
            }
        )

    aligned_active = align_equity_curves(
        strategy_aligned,
        benchmark_gross,
    )

    active_metrics = active_performance_metrics(
        aligned_active
    )

    active_metrics.update(
        {
            "Period": period_name,
            "Strategy_CAGR_%": strategy_metrics["CAGR_%"],
            "BIST100_CAGR_%": gross_metrics["CAGR_%"],
            "Excess_CAGR_pp": (
                strategy_metrics["CAGR_%"]
                - gross_metrics["CAGR_%"]
            ),
            "Strategy_Max_DD_%": strategy_metrics[
                "Max_Drawdown_%"
            ],
            "BIST100_Max_DD_%": gross_metrics[
                "Max_Drawdown_%"
            ],
            "Drawdown_Difference_pp": (
                strategy_metrics["Max_Drawdown_%"]
                - gross_metrics["Max_Drawdown_%"]
            ),
        }
    )

    active_records.append(active_metrics)

    stored_outputs[period_name] = {
        "strategy_equity": strategy_aligned,
        "strategy_trades": strategy_trades,
        "benchmark_gross": benchmark_gross,
        "benchmark_net": benchmark_net,
        "aligned_active": aligned_active,
    }

comparison_table = pd.DataFrame(period_records)
active_table = pd.DataFrame(active_records)

display(
    comparison_table.sort_values(
        ["Period", "Portfolio"]
    )
)

print("AKTİF PERFORMANS")
display(
    active_table.sort_values("Period")
)


## 3. Dönem bazında özet görünüm


In [ ]:
summary_view = comparison_table.pivot_table(
    index="Period",
    columns="Portfolio",
    values=[
        "End_Value_TL",
        "Total_Return_%",
        "CAGR_%",
        "Max_Drawdown_%",
        "Sharpe",
        "Calmar",
    ],
    aggfunc="first",
)

display(summary_view)


## 4. Tam dönem: 500.000 TL'nin gelişimi


In [ ]:
full_strategy = stored_outputs["Full"][
    "strategy_equity"
]
full_gross = stored_outputs["Full"][
    "benchmark_gross"
]
full_net = stored_outputs["Full"][
    "benchmark_net"
]

growth_chart = (
    full_strategy[["Date", "Equity"]]
    .rename(
        columns={"Equity": "Final Strategy"}
    )
    .merge(
        full_gross[["Date", "Equity"]].rename(
            columns={"Equity": "BIST100 Gross"}
        ),
        on="Date",
        how="inner",
    )
    .merge(
        full_net[["Date", "Equity"]].rename(
            columns={"Equity": "BIST100 Net"}
        ),
        on="Date",
        how="inner",
    )
)

growth_chart = growth_chart.set_index("Date")

plt.figure(figsize=(13, 7))
plt.plot(
    growth_chart.index,
    growth_chart["Final Strategy"],
    label="Final Strategy",
)
plt.plot(
    growth_chart.index,
    growth_chart["BIST100 Gross"],
    label="BIST100 Gross",
)
plt.plot(
    growth_chart.index,
    growth_chart["BIST100 Net"],
    label="BIST100 Net",
)
plt.title("500.000 TL — Final Strateji ve BIST100")
plt.xlabel("Tarih")
plt.ylabel("Portföy Değeri (TL)")
plt.legend()
plt.tight_layout()
plt.show()


## 5. Tam dönem drawdown karşılaştırması


In [ ]:
strategy_dd = drawdown_series(
    full_strategy
).rename(
    columns={"Drawdown_%": "Final Strategy"}
)

bist100_dd = drawdown_series(
    full_gross
).rename(
    columns={"Drawdown_%": "BIST100 Gross"}
)

drawdown_chart = strategy_dd.merge(
    bist100_dd,
    on="Date",
    how="inner",
).set_index("Date")

plt.figure(figsize=(13, 6))
plt.plot(
    drawdown_chart.index,
    drawdown_chart["Final Strategy"],
    label="Final Strategy",
)
plt.plot(
    drawdown_chart.index,
    drawdown_chart["BIST100 Gross"],
    label="BIST100 Gross",
)
plt.axhline(0, linewidth=1)
plt.title("Drawdown Karşılaştırması")
plt.xlabel("Tarih")
plt.ylabel("Drawdown (%)")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Takvim yılı getirileri


In [ ]:
yearly_returns = calendar_return_table(
    {
        "Final Strategy": full_strategy,
        "BIST100 Gross": full_gross,
    },
    frequency="YE",
)

yearly_returns["Year"] = (
    yearly_returns["Date"].dt.year
)

yearly_table = yearly_returns.pivot_table(
    index="Year",
    columns="Portfolio",
    values="Return_%",
    aggfunc="first",
).reset_index()

yearly_table["Excess_Return_pp"] = (
    yearly_table["Final Strategy"]
    - yearly_table["BIST100 Gross"]
)

display(yearly_table)


In [ ]:
yearly_plot = yearly_table.set_index("Year")[
    ["Final Strategy", "BIST100 Gross"]
]

plt.figure(figsize=(12, 6))
yearly_plot.plot(
    kind="bar",
    ax=plt.gca(),
)
plt.axhline(0, linewidth=1)
plt.title("Takvim Yılı Getirileri")
plt.xlabel("Yıl")
plt.ylabel("Getiri (%)")
plt.tight_layout()
plt.show()


## 7. Aylık başarı oranı ve rolling 12 aylık getiri


In [ ]:
monthly_returns = calendar_return_table(
    {
        "Final Strategy": full_strategy,
        "BIST100 Gross": full_gross,
    },
    frequency="ME",
)

monthly_pivot = monthly_returns.pivot_table(
    index="Date",
    columns="Portfolio",
    values="Return_%",
    aggfunc="first",
).dropna()

monthly_pivot["Excess_Return_pp"] = (
    monthly_pivot["Final Strategy"]
    - monthly_pivot["BIST100 Gross"]
)

monthly_summary = pd.DataFrame(
    [
        {
            "Compared_Months": len(monthly_pivot),
            "Strategy_Beat_BIST100_Months": int(
                monthly_pivot[
                    "Excess_Return_pp"
                ].gt(0).sum()
            ),
            "Monthly_Hit_Rate_%": (
                monthly_pivot[
                    "Excess_Return_pp"
                ].gt(0).mean()
                * 100
            ),
            "Average_Monthly_Excess_pp": (
                monthly_pivot[
                    "Excess_Return_pp"
                ].mean()
            ),
            "Median_Monthly_Excess_pp": (
                monthly_pivot[
                    "Excess_Return_pp"
                ].median()
            ),
        }
    ]
)

display(monthly_summary)


In [ ]:
rolling_returns = growth_chart.pct_change(
    252,
    fill_method=None,
) * 100

plt.figure(figsize=(13, 6))
plt.plot(
    rolling_returns.index,
    rolling_returns["Final Strategy"],
    label="Final Strategy",
)
plt.plot(
    rolling_returns.index,
    rolling_returns["BIST100 Gross"],
    label="BIST100 Gross",
)
plt.axhline(0, linewidth=1)
plt.title("Rolling 252 İşlem Günü Getirisi")
plt.xlabel("Tarih")
plt.ylabel("Getiri (%)")
plt.legend()
plt.tight_layout()
plt.show()


## 8. Sonuçları kaydet

`Final Strategy` sonuçları komisyon ve slippage içerir. `BIST100 Gross`
standart fiyat endeksi karşılaştırmasıdır. `BIST100 Net` aynı maliyetlerle
tek seferlik al-tut uygulamasını gösterir.


In [ ]:
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

comparison_table.to_csv(
    RESULTS_DIR
    / "final_strategy_vs_bist100_periods.csv",
    index=False,
)

active_table.to_csv(
    RESULTS_DIR
    / "final_strategy_vs_bist100_active_metrics.csv",
    index=False,
)

yearly_table.to_csv(
    RESULTS_DIR
    / "final_strategy_vs_bist100_yearly.csv",
    index=False,
)

monthly_pivot.reset_index().to_csv(
    RESULTS_DIR
    / "final_strategy_vs_bist100_monthly.csv",
    index=False,
)

monthly_summary.to_csv(
    RESULTS_DIR
    / "final_strategy_vs_bist100_monthly_summary.csv",
    index=False,
)

full_strategy.to_parquet(
    RESULTS_DIR
    / "final_strategy_equity.parquet",
    index=False,
)

stored_outputs["Full"][
    "strategy_trades"
].to_csv(
    RESULTS_DIR
    / "final_strategy_trades.csv",
    index=False,
)

full_gross.to_parquet(
    RESULTS_DIR
    / "bist100_gross_benchmark_equity.parquet",
    index=False,
)

full_net.to_parquet(
    RESULTS_DIR
    / "bist100_net_benchmark_equity.parquet",
    index=False,
)

print("BIST100 karşılaştırma sonuçları kaydedildi.")


## Yorumlama notları

- **Excess CAGR pozitifse:** Strateji yıllıklandırılmış olarak BIST100'ü geçmiş.
- **Alpha pozitifse:** BIST100 günlük hareketleriyle açıklanamayan ek getiri var.
- **Beta 1'in altındaysa:** Strateji BIST100'e göre daha düşük piyasa duyarlılığı taşıyor.
- **Downside Capture %100'ün altındaysa:** BIST100 düşüş günlerinin daha küçük bir kısmı portföye yansımış.
- **Information Ratio pozitif ve yüksekse:** Aktif getiri daha tutarlı üretilmiş.

Sınırlamalar:

1. BIST100 (`XU100.IS`) bir fiyat endeksidir; temettülerin tamamını temsil eden
   toplam getiri endeksi değildir.
2. Hisse fiyatlarında `auto_adjust=True` kullanılması temettü ve bölünme
   düzeltmelerini içerebilir; bu nedenle kıyaslama tamamen bire bir değildir.
3. Güncel BIST100 listesini geçmişe uygulamak survivorship bias oluşturur.
4. Development, validation ve holdout sonuçları artık görülmüştür. Gerçek
   yeni out-of-sample değerlendirme paper trading ile yapılacaktır.
